In [1]:
import pandas as pd
import yaml
from pathlib import Path
import argparse
from datetime import datetime, timedelta

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [8]:


df_procesar = pd.read_csv(
    "../data/raw/ventas_completo.csv",
    usecols=['identification_doct', 'product', 'date_sale'])
# Renombrar y filtrar IDs de cliente
df_procesar['client'] = df_procesar['identification_doct'].astype(str).str.strip()
mask_digits = df_procesar["client"].str.isdigit().fillna(False)
mask_zero = ~df_procesar["client"].str.startswith("0", na=False)
df_procesar = df_procesar[mask_digits & mask_zero].copy()
print(f"Filas tras filtrar id_client no numéricos/cero inicial: {len(df_procesar)}")

# Limpiar fechas, productos domicilio
df_procesar['product'] = df_procesar['product'].astype(str).str.strip()
df_procesar['date_sale'] = pd.to_datetime(df_procesar['date_sale'])
df_procesar = df_procesar.dropna(subset=['date_sale'])
df_procesar['date_sale'] = df_procesar['date_sale'].dt.normalize()


productos = pd.read_csv(
    f"../data/raw/productos.csv",
    dtype={"codigo_unico": str}
).rename(columns={"codigo_unico": "product"})
productos = productos.loc[:, ["product", "description", "brand", "category"]]
productos = productos.drop_duplicates("product")
productos['product'] = productos['product'].str.strip()
print(f"Datos cargados Filas: {len(df_procesar)}")


predictions = pd.read_csv("../predictions/predictions.csv", converters={"client": str})

df_procesar = pd.merge(df_procesar, productos, on="product")

/var/folders/1g/77kw2x4j5678s_87_sqc1fpc0000gp/T/ipykernel_41821/1079516404.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_procesar = pd.read_csv(


Filas tras filtrar id_client no numéricos/cero inicial: 18432076
Datos cargados Filas: 18432076


In [40]:
last_month = 1

print(f"Filtrando clientes con compras en los últimos {last_month} meses...")
last_months_ago = datetime.now().date() - timedelta(days=last_month*30)
recent_purchases = df_procesar[df_procesar["date_sale"].dt.date >= last_months_ago]
print(f"Compras en los últimos {last_month} meses: {len(recent_purchases)}")
hist_product_client = df_procesar[df_procesar["client"].isin(predictions["client"])]
print(f"Clientes predichos con compras en los últimos {last_month} meses: {hist_product_client["client"].nunique()}")


Filtrando clientes con compras en los últimos 1 meses...
Compras en los últimos 1 meses: 2606988
Clientes predichos con compras en los últimos 1 meses: 459


In [41]:

# Paso 1: contar frecuencia relativa por cliente y producto
freq = (
    hist_product_client.groupby(['client', 'description'])
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum())
    .reset_index(name='rel_freq')
)

# Paso 2: ordenar por cliente y frecuencia relativa descendente
freq_sorted = freq.sort_values(['client', 'rel_freq'], ascending=[True, False])

# Paso 3: obtener el top 25% de productos más frecuentes por cliente
def top_percent(group):
    n = max(1, int(len(group) * 0.25))  # al menos 1
    return group.head(n)

top_products = freq_sorted.groupby('client', group_keys=False).apply(top_percent)

# Resultado: top_products contiene para cada cliente el 25% más frecuente de productos


In [42]:
top_products[top_products["client"] == '1005184871']

,client,description,rel_freq
8800,1005184871,GASEOSA PEPSI PET *400ML,0.350000
8798,1005184871,GASEOSA COCACOLA *1.5L,0.083333
8799,1005184871,GASEOSA COCACOLA MEGA x 2.5L,0.050000
8790,1005184871,CHOCORAMO TAJADA RAMO * 65 GR,0.033333
8792,1005184871,COCACOLA FLE x I x 400ML,0.033333
8794,1005184871,DEL VALLE CITRUS PET* 1.5 LT,0.033333
8795,1005184871,DETODITO BBQ*165G,0.033333


In [43]:
df_procesar[df_procesar["client"] == '1005184871']["description"].value_counts(normalize=True)

GASEOSA PEPSI PET *400ML                    0.350000
GASEOSA COCACOLA  *1.5L                     0.083333
GASEOSA COCACOLA MEGA  x 2.5L               0.050000
TOCINO CARNE. QB                            0.033333
DETODITO BBQ*165G                           0.033333
PASABOCAS CHEETOS TRISSITOS PICANTE*34GR    0.033333
DEL VALLE CITRUS PET* 1.5 LT                0.033333
COCACOLA FLE x I  x  400ML                  0.033333
CHOCORAMO TAJADA RAMO * 65 GR               0.033333
YOGURT ALQUERIA CEREAL FLIPS x 170GR        0.016667
PONQUESITO CON GOTAS CHOC BIMB              0.016667
REFRESCO HIT MORA *1500ML                   0.016667
HELADO COOKIES & CREAM CREM HELADO*300GR    0.016667
TAKIS FUEGO *50GR                           0.016667
EMPANADA PAPA Y CARNE *60GR                 0.016667
PAN CHINO   x 6UNDS                         0.016667
PONY MALTA PET  x 1.5LT                     0.016667
PAPA MARGARITA POLLO x 105GR                0.016667
BIMBOLETES  x 2  x 55GR                     0.

In [44]:
df_procesar[df_procesar["client"] == '1005184871']["date_sale"].min()

Timestamp('2025-02-20 00:00:00')

In [45]:
top_summary = (
    top_products
    .groupby('client')
    .agg(
        productos_comunes=('description', lambda x: ', '.join(x)),
        cantidad_productos=('description', 'count')
    )
    .reset_index()
)
top_summary

,client,productos_comunes,cantidad_productos
0,1000192661,"BUNUELO EURO 55 gr, JUGO NATURAL X 12 OZ MENU,...",51
1,1000410009,"BUNUELO EURO 55 gr, COCACOLA FLE x I x 400ML...",59
2,1000440285,"GASEOSA COCACOLA *1.5L, BUNUELO EURO 55 gr, M...",65
3,1000445397,"ENERGIZANTE AMPER PREDATOR *473ML, BEBIDA ENER...",48
4,1000534537,"MR TEA LIMON PET x 500ML, AGUA LIMA/LIMON HOJ...",36
...,...,...,...
454,98544040,"BOLSA PLASTICA, LECHE COLANTA SEMIDESCREM 1000...",29
455,98552891,"AGUA POTABLE TRATADA EUROMAX *1000ML, HALLS LI...",47
456,98558712,"AREPA TELA BLANCA CASERA SARY X10 UNDS, QUESIT...",55
457,98630133,"AGUA BRISA LIMA LIMON PET*1500ML, AGUA H2O LIM...",17


In [46]:
predictions_with_precom = pd.merge(predictions, top_summary, on="client", how="left")

In [47]:
predictions_with_precom

,date,client,prob,name,email,phone,telephone,productos_comunes,cantidad_productos
0,2025-05-15,70569726,0.7463,OTALVARO GONZALEZ LUIS FERNANDO,luisfernandootalvarogonzalez@hotmail.com,3.108321e+09,0.000000e+00,"BOLSA PLASTICA, CIGARRILLO CHESTERFIRLD BLUE B...",86
1,2025-05-15,800180330,0.7463,COMPAÑIA DE ALIMENTOS COLOMBIANOS CALCO S.A,proveedores.fe.medellin@crepesywaffles.com,3.102119e+09,0.000000e+00,"PLATANO, PAPA CAPIRA KILO, PAPA CRIOLLA, PULPA...",135
2,2025-05-15,1235245925,0.7350,GISEL FERNANDEZ,gisellecfr@gmail.com,3.013476e+09,0.000000e+00,"ARROZ EURO x 5 LIBRAS, HARINA DE TRIGO FROT ...",59
3,2025-05-15,7797719,0.7350,BUTSU WAKE,wake.butsu@gmail.com,3.136017e+09,NaN,"BOLSA PLASTICA, AGUACATE HASS, ZANAHORIA, PLAT...",39
4,2025-05-15,901569974,0.7350,PATAKUS FOOD SAS,facturaspatakusfoodsas@gmail.com,3.015186e+09,0.000000e+00,"TRANSPORTE DOMICILIO, HUESO DE RES CARNUDO, CO...",90
...,...,...,...,...,...,...,...,...,...
454,2025-05-15,1036956908,0.5017,CONTRERAS HOYOS LENIS,lenicontrerashoyos@hotmail.com,3.132863e+09,3.132863e+09,"MANDARINA ONECA, MANI KRAKS LA ESPECIAL *38 GR...",42
455,2025-05-15,1140887977,0.5017,GONZALEZ ADISON,adisongonzalezba@gmail.com,3.246062e+09,0.000000e+00,"MENU CON POLLO EURO, BEBIDA ENERGIZANTE AMPER ...",20
456,2025-05-15,34967510,0.5008,NEGRETE YENI,facturacionelectronicapos@eurosupermercados.com,NaN,0.000000e+00,"BOLSA PLASTICA, TOMATE CHONTO, ZANAHORIA, PECH...",69
457,2025-05-15,43688000,0.5008,PAULA RESTREPO,facturacionelectronicapos@eurosupermercados.com,NaN,2.786382e+06,"PANELA EUROMAX x 8 UND x 812 GR, TOSTADA SE...",69
